In [0]:
DESCRIBE meridian_dev.bronze.encounters;

In [0]:
DESCRIBE meridian_dev.bronze.conditions;

In [0]:
%python
from pyspark.sql.functions import col, datediff, year, floor, months_between, current_date

enc = spark.read.table("meridian_dev.bronze.encounters")
pat = spark.read.table("meridian_dev.bronze.patients")

# --- SAFE patient attributes only: NO ssn, names, address, lat/lon ---
# We bring generalized/analytical columns: gender, birthdate (→ age), state, race/ethnicity
pat_safe = pat.select(
    col("Id").alias("patient_id"),
    col("GENDER").alias("gender"),
    col("BIRTHDATE").alias("birthdate"),
    col("STATE").alias("state"),
    col("RACE").alias("race"),
    col("ETHNICITY").alias("ethnicity"),
)

silver_encounters = (enc
    .withColumnRenamed("Id", "encounter_id")
    .withColumnRenamed("PATIENT", "patient_id")
    # computed: encounter duration in minutes
    .withColumn("duration_min",
        (col("STOP").cast("long") - col("START").cast("long")) / 60)
    # join in ONLY the safe patient attributes
    .join(pat_safe, on="patient_id", how="left")
    # derive age at time of encounter (generalized — not raw birthdate)
    .withColumn("age_at_encounter",
        floor(months_between(col("START"), col("birthdate")) / 12))
    # select a clean, analytical column set — note: NO PHI columns present
    .select(
        "encounter_id", "patient_id",
        col("START").alias("encounter_start"),
        col("STOP").alias("encounter_stop"),
        "duration_min",
        col("ENCOUNTERCLASS").alias("encounter_class"),
        col("DESCRIPTION").alias("encounter_desc"),
        col("PAYER").alias("payer_id"),
        col("BASE_ENCOUNTER_COST").alias("base_cost"),
        col("TOTAL_CLAIM_COST").alias("total_cost"),
        col("PAYER_COVERAGE").alias("payer_coverage"),
        col("REASONDESCRIPTION").alias("reason_desc"),
        # safe patient attributes
        "gender", "age_at_encounter", "state", "race", "ethnicity",
    ))

silver_encounters.write.format("delta").mode("overwrite") \
    .saveAsTable("meridian_dev.silver.encounters")

print(f"Silver encounters built: {silver_encounters.count()} rows")

In [0]:
SELECT encounter_class, count(*) AS n,
       round(avg(total_cost), 2) AS avg_cost,
       round(avg(age_at_encounter), 1) AS avg_age
FROM meridian_dev.silver.encounters
GROUP BY encounter_class
ORDER BY n DESC;

In [0]:
SELECT
    round(min(total_cost), 2)  AS min_total,
    round(max(total_cost), 2)  AS max_total,
    round(avg(total_cost), 2)  AS avg_total,
    round(min(base_cost), 2)   AS min_base,
    round(max(base_cost), 2)   AS max_base
FROM meridian_dev.silver.encounters;

In [0]:
DESCRIBE meridian_dev.bronze.medications;

In [0]:
DESCRIBE meridian_dev.bronze.procedures;

In [0]:
SELECT
    round(min(TOTALCOST), 2) AS min_med, round(max(TOTALCOST), 2) AS max_med,
    round(avg(TOTALCOST), 2) AS avg_med
FROM meridian_dev.bronze.medications;